# Document Reader 10% Experiment: SQuAD Context -> Answer Span / No Answer

This notebook runs the first 10% SQuAD v2 experiment. Retrieval is skipped for now: the reader receives each SQuAD 2.0 question together with the SQuAD `context` field as the passage.

Pipeline: `SQuAD question + SQuAD context -> Document Reader -> answer span or no answer`

In [49]:
# Colab setup. Runtime > Change runtime type > GPU is recommended.
!pip -q install transformers datasets evaluate accelerate


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [50]:
import collections
import os
import random
import time

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, load_dataset
from torch import nn
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    BertModel,
    BertPreTrainedModel,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
from transformers.modeling_outputs import QuestionAnsweringModelOutput

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())

Torch: 2.13.0
CUDA available: False
MPS available: True


## Configuration

This run uses a 10% SQuAD v2 subset as the first real experiment after the smoke test. It keeps the same reader pipeline, post-processing, SQuAD v2 scoring, and qualitative inspection table.

In [51]:
MODEL_NAME = "bert-base-uncased"
SEED = 42
MAX_LENGTH = 384
DOC_STRIDE = 128
SUBSET_FRACTION = 0.01
OUTPUT_DIR = "./reader_1pct_outputs"

TRAINING_HYPERPARAMS = dict(
    learning_rate=3e-5,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load SQuAD 2.0 and Add a Replaceable Passage Provider

For now, `passage == context`. Later, only `add_reader_passages` needs to change when retrieval starts producing passages.

In [52]:
raw_squad = load_dataset("rajpurkar/squad_v2")

train_example_count = int(len(raw_squad["train"]) * SUBSET_FRACTION)
validation_example_count = int(len(raw_squad["validation"]) * SUBSET_FRACTION)

train_smoke = raw_squad["train"].shuffle(seed=SEED).select(range(train_example_count))
validation_smoke = raw_squad["validation"].shuffle(seed=SEED).select(range(validation_example_count))
smoke_data = DatasetDict({"train": train_smoke, "validation": validation_smoke})


def add_reader_passages(batch):
    return {"passage": batch["context"]}

smoke_data = smoke_data.map(add_reader_passages, batched=True)
smoke_data

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'passage'],
        num_rows: 1303
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'passage'],
        num_rows: 118
    })
})

## Tokenization and Labels

This is the standard long-context QA setup: `max_length=384`, `stride=128`, overflow windows, and `[CLS]` labels for no-answer examples or answer spans outside a given window.

In [53]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def prepare_train_features(examples):
    tokenized = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["passage"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    special_tokens_mask = tokenized.pop("special_tokens_mask")

    start_positions, end_positions, example_ids = [], [], []
    question_token_mask, context_token_mask = [], []

    for feature_index, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][feature_index]
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        answers = examples["answers"][sample_index]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        example_ids.append(examples["id"][sample_index])

        question_token_mask.append([
            int(sequence_ids[i] == 0 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(input_ids))
        ])
        context_token_mask.append([
            int(sequence_ids[i] == 1 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(input_ids))
        ])

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= answer_start and offsets[token_end_index][1] >= answer_end):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= answer_start:
                token_start_index += 1
            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1
            start_positions.append(token_start_index - 1)
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized["example_id"] = example_ids
    tokenized["question_token_mask"] = question_token_mask
    tokenized["context_token_mask"] = context_token_mask
    return tokenized


def prepare_validation_features(examples):
    tokenized = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["passage"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    special_tokens_mask = tokenized.pop("special_tokens_mask")
    example_ids, question_token_mask, context_token_mask = [], [], []

    for feature_index in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        example_ids.append(examples["id"][sample_index])
        question_token_mask.append([
            int(sequence_ids[i] == 0 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(tokenized["input_ids"][feature_index]))
        ])
        context_token_mask.append([
            int(sequence_ids[i] == 1 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(tokenized["input_ids"][feature_index]))
        ])
        tokenized["offset_mapping"][feature_index] = [
            offset if sequence_ids[i] == 1 else None
            for i, offset in enumerate(tokenized["offset_mapping"][feature_index])
        ]

    tokenized["example_id"] = example_ids
    tokenized["question_token_mask"] = question_token_mask
    tokenized["context_token_mask"] = context_token_mask
    return tokenized

train_features = smoke_data["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=smoke_data["train"].column_names,
)
validation_features = smoke_data["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=smoke_data["validation"].column_names,
)

print(train_features)
print(validation_features)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions', 'example_id', 'question_token_mask', 'context_token_mask'],
    num_rows: 1324
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id', 'question_token_mask', 'context_token_mask'],
    num_rows: 120
})


## Custom BERT + DrQA-Inspired Attention Reader

This model separates question and context tokens, aligns each context token to a soft attention-weighted question representation, combines context/question/interaction features, projects back to BERT hidden size, and predicts start/end logits. `[CLS]` remains the no-answer position.

In [54]:
class BertDrQAQuestionAttentionForQA(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.similarity = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.projection = nn.Sequential(
            nn.Linear(config.hidden_size * 3, config.hidden_size),
            nn.GELU(),
            nn.LayerNorm(config.hidden_size),
            nn.Dropout(config.hidden_dropout_prob),
        )
        self.qa_outputs = nn.Linear(config.hidden_size, 2)
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        start_positions=None,
        end_positions=None,
        question_token_mask=None,
        context_token_mask=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]

        if question_token_mask is None:
            question_token_mask = ((token_type_ids == 0) & (attention_mask == 1)).long()
        if context_token_mask is None:
            context_token_mask = ((token_type_ids == 1) & (attention_mask == 1)).long()

        q_mask = question_token_mask.bool()
        c_mask = context_token_mask.bool()

        similarity_scores = torch.matmul(self.similarity(sequence_output), sequence_output.transpose(1, 2))
        similarity_scores = similarity_scores.masked_fill(~q_mask[:, None, :], torch.finfo(similarity_scores.dtype).min)
        attention_weights = torch.softmax(similarity_scores, dim=-1)
        aligned_question = torch.matmul(attention_weights, sequence_output)

        combined = torch.cat([sequence_output, aligned_question, sequence_output * aligned_question], dim=-1)
        enhanced_output = self.projection(combined)

        valid_answer_positions = c_mask.clone()
        valid_answer_positions[:, 0] = True
        enhanced_output = torch.where(valid_answer_positions[:, :, None], enhanced_output, sequence_output)

        logits = self.qa_outputs(enhanced_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1).contiguous()
        end_logits = end_logits.squeeze(-1).contiguous()

        invalid_positions = ~valid_answer_positions
        start_logits = start_logits.masked_fill(invalid_positions, torch.finfo(start_logits.dtype).min)
        end_logits = end_logits.masked_fill(invalid_positions, torch.finfo(end_logits.dtype).min)

        total_loss = None
        if start_positions is not None and end_positions is not None:
            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)
            loss_fct = nn.CrossEntropyLoss(ignore_index=ignored_index)
            total_loss = (loss_fct(start_logits, start_positions) + loss_fct(end_logits, end_positions)) / 2

        if not return_dict:
            output = (start_logits, end_logits) + outputs[2:]
            return ((total_loss,) + output) if total_loss is not None else output

        return QuestionAnsweringModelOutput(
            loss=total_loss,
            start_logits=start_logits,
            end_logits=end_logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

## Post-Processing and Metrics

This converts token logits back to text spans and uses the official SQuAD v2 metric. It also reports answerability accuracy, precision, recall, and F1.

In [55]:
squad_v2_metric = evaluate.load("squad_v2")


def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = collections.OrderedDict()
    for example_index, example in enumerate(examples):
        min_null_score = None
        valid_answers = []
        context = example["passage"]

        for feature_index in features_per_example[example_index]:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            cls_score = start_logits[0] + end_logits[0]
            min_null_score = cls_score if min_null_score is None else min(min_null_score, cls_score)

            start_indexes = np.argsort(start_logits)[-1:-n_best_size - 1:-1].tolist()
            end_indexes = np.argsort(end_logits)[-1:-n_best_size - 1:-1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue
                    if offset_mapping[start_index] is None or offset_mapping[end_index] is None:
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue
                    start_char, _ = offset_mapping[start_index]
                    _, end_char = offset_mapping[end_index]
                    valid_answers.append({
                        "score": start_logits[start_index] + end_logits[end_index],
                        "text": context[start_char:end_char],
                    })

        best_answer = max(valid_answers, key=lambda x: x["score"]) if valid_answers else {"text": "", "score": 0.0}
        predictions[example["id"]] = "" if min_null_score is not None and min_null_score > best_answer["score"] else best_answer["text"]

    formatted_predictions = [
        {"id": example_id, "prediction_text": text, "no_answer_probability": float(text == "")}
        for example_id, text in predictions.items()
    ]
    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return formatted_predictions, references


def compute_squad_v2_metrics(predictions, references):
    overall = squad_v2_metric.compute(predictions=predictions, references=references)
    refs_by_id = {ref["id"]: ref for ref in references}
    ans_preds, ans_refs, no_preds, no_refs = [], [], [], []
    true_positive = false_positive = true_negative = false_negative = 0
    for pred in predictions:
        ref = refs_by_id[pred["id"]]
        gold_answerable = len(ref["answers"]["text"]) > 0
        predicted_answerable = pred["prediction_text"] != ""

        if gold_answerable and predicted_answerable:
            true_positive += 1
        elif not gold_answerable and predicted_answerable:
            false_positive += 1
        elif not gold_answerable and not predicted_answerable:
            true_negative += 1
        else:
            false_negative += 1

        if not gold_answerable:
            no_preds.append(pred); no_refs.append(ref)
        else:
            ans_preds.append(pred); ans_refs.append(ref)
    answerable = squad_v2_metric.compute(predictions=ans_preds, references=ans_refs) if ans_preds else {}
    unanswerable = squad_v2_metric.compute(predictions=no_preds, references=no_refs) if no_preds else {}
    total = true_positive + false_positive + true_negative + false_negative
    answerability_accuracy = (true_positive + true_negative) / total * 100 if total else 0.0
    answerability_precision = true_positive / (true_positive + false_positive) * 100 if true_positive + false_positive else 0.0
    answerability_recall = true_positive / (true_positive + false_negative) * 100 if true_positive + false_negative else 0.0
    answerability_f1 = (
        2 * answerability_precision * answerability_recall / (answerability_precision + answerability_recall)
        if answerability_precision + answerability_recall else 0.0
    )
    return {
        "overall_em": overall.get("exact", 0.0),
        "overall_f1": overall.get("f1", 0.0),
        "answerability_accuracy": answerability_accuracy,
        "answerability_precision": answerability_precision,
        "answerability_recall": answerability_recall,
        "answerability_f1": answerability_f1,
        "answerable_em": answerable.get("exact", 0.0),
        "answerable_f1": answerable.get("f1", 0.0),
        "unanswerable_em": unanswerable.get("exact", 0.0),
        "unanswerable_f1": unanswerable.get("f1", 0.0),
        "answerability_tp": true_positive,
        "answerability_fp": false_positive,
        "answerability_tn": true_negative,
        "answerability_fn": false_negative,
    }


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Run Both 10% Experiments

Both models use identical 10% data and hyperparameters so the baseline BERT reader and DrQA-style attention reader can be compared directly.

In [56]:
smoke_predictions_by_model = {}
smoke_references_by_id = {}

def build_model(model_kind):
    if model_kind == "bert_baseline":
        return AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
    if model_kind == "bert_drqa_attention":
        return BertDrQAQuestionAttentionForQA.from_pretrained(MODEL_NAME)
    raise ValueError(f"Unknown model kind: {model_kind}")


def run_reader_smoke_test(model_kind):
    print(f"\n=== Running {model_kind} ===")
    model = build_model(model_kind)
    params = count_trainable_parameters(model)

    train_dataset = train_features.remove_columns(["example_id"])
    eval_dataset = validation_features.remove_columns(["example_id", "offset_mapping"])

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_kind),
        seed=SEED,
        data_seed=SEED,
        **TRAINING_HYPERPARAMS,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=default_data_collator,
    )

    start = time.perf_counter()
    trainer.train()
    train_time_sec = time.perf_counter() - start

    start = time.perf_counter()
    raw_predictions = trainer.predict(eval_dataset).predictions
    inference_time_sec = time.perf_counter() - start

    predictions, references = postprocess_qa_predictions(smoke_data["validation"], validation_features, raw_predictions)
    smoke_predictions_by_model[model_kind] = {pred["id"]: pred for pred in predictions}
    smoke_references_by_id.update({ref["id"]: ref for ref in references})
    metrics = compute_squad_v2_metrics(predictions, references)

    result = {
        "model": model_kind,
        "train_examples": len(smoke_data["train"]),
        "validation_examples": len(smoke_data["validation"]),
        "parameters": params,
        "train_time_sec": round(train_time_sec, 2),
        "inference_time_sec": round(inference_time_sec, 2),
        **{k: round(v, 2) for k, v in metrics.items()},
    }
    print(result)
    return result

smoke_results = [run_reader_smoke_test(kind) for kind in ["bert_baseline", "bert_drqa_attention"]]
pd.DataFrame(smoke_results)


=== Running bert_baseline ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4719.81it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

Step,Training Loss
100,3.975500
200,2.695438
300,2.117454


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.44s/it]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.52s/it]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'model': 'bert_baseline', 'train_examples': 1303, 'validation_examples': 118, 'parameters': 108893186, 'train_time_sec': 1012.02, 'inference_time_sec': 11.99, 'overall_em': 50.85, 'overall_f1': 51.0, 'answerability_accuracy': 54.24, 'answerability_precision': 62.5, 'answerability_recall': 17.24, 'answerability_f1': 27.03, 'answerable_em': 10.34, 'answerable_f1': 10.66, 'unanswerable_em': 90.0, 'unanswerable_f1': 90.0, 'answerability_tp': 10, 'answerability_fp': 6, 'answerability_tn': 54, 'answerability_fn': 48}

=== Running bert_drqa_attention ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5289.46it/s]
[transformers] BertDrQAQuestionAttentionForQA LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
projection.{0, 2}.bias                     | MISSING    | 
similarity.weight                          | MISSING    | 
projection.{0, 2}.weigh

Step,Training Loss
100,3.707303
200,2.596165
300,1.994626


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.03it/s]
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'model': 'bert_drqa_attention', 'train_examples': 1303, 'validation_examples': 118, 'parameters': 111254786, 'train_time_sec': 619.74, 'inference_time_sec': 12.72, 'overall_em': 46.61, 'overall_f1': 48.06, 'answerability_accuracy': 54.24, 'answerability_precision': 56.67, 'answerability_recall': 29.31, 'answerability_f1': 38.64, 'answerable_em': 13.79, 'answerable_f1': 16.74, 'unanswerable_em': 78.33, 'unanswerable_f1': 78.33, 'answerability_tp': 17, 'answerability_fp': 13, 'answerability_tn': 47, 'answerability_fn': 41}


,model,train_examples,validation_examples,parameters,train_time_sec,inference_time_sec,overall_em,overall_f1,answerability_accuracy,answerability_precision,answerability_recall,answerability_f1,answerable_em,answerable_f1,unanswerable_em,unanswerable_f1,answerability_tp,answerability_fp,answerability_tn,answerability_fn
0,bert_baseline,1303,118,108893186,1012.02,11.99,50.85,51.00,54.24,62.50,17.24,27.03,10.34,10.66,90.00,90.00,10,6,54,48
1,bert_drqa_attention,1303,118,111254786,619.74,12.72,46.61,48.06,54.24,56.67,29.31,38.64,13.79,16.74,78.33,78.33,17,13,47,41


## Qualitative Test on 10 Random SQuAD Questions

This samples 10 validation examples and compares the gold SQuAD answer against both 10%-trained readers. These are qualitative checks, not final accuracy numbers.

In [57]:
def normalize_text_for_exact_match(text):
    return " ".join(str(text).lower().strip().split())


def gold_answer_texts(example):
    answers = example["answers"]["text"]
    return answers if answers else [""]


def exact_match_any_gold(prediction_text, gold_texts):
    normalized_prediction = normalize_text_for_exact_match(prediction_text)
    return any(normalized_prediction == normalize_text_for_exact_match(gold) for gold in gold_texts)


rng = random.Random(SEED)
sample_count = min(10, len(smoke_data["validation"]))
sample_indices = rng.sample(range(len(smoke_data["validation"])), sample_count)

qualitative_rows = []
for row_number, example_index in enumerate(sample_indices, start=1):
    example = smoke_data["validation"][example_index]
    example_id = example["id"]
    gold_texts = gold_answer_texts(example)
    bert_prediction = smoke_predictions_by_model["bert_baseline"][example_id]["prediction_text"]
    drqa_prediction = smoke_predictions_by_model["bert_drqa_attention"][example_id]["prediction_text"]
    qualitative_rows.append({
        "#": row_number,
        "id": example_id,
        "question": example["question"],
        "gold_answers": " | ".join(gold_texts) if gold_texts != [""] else "[no answer]",
        "bert_prediction": bert_prediction if bert_prediction else "[no answer]",
        "bert_exact": exact_match_any_gold(bert_prediction, gold_texts),
        "drqa_prediction": drqa_prediction if drqa_prediction else "[no answer]",
        "drqa_exact": exact_match_any_gold(drqa_prediction, gold_texts),
        "passage_preview": example["passage"][:240] + ("..." if len(example["passage"]) > 240 else ""),
    })

qualitative_df = pd.DataFrame(qualitative_rows)
qualitative_df


,#,id,question,gold_answers,bert_prediction,bert_exact,drqa_prediction,drqa_exact,passage_preview
0,1,571a4b0f10f8ca1400304fd7,What is consumed in both combustion and respir...,nitroaereus | nitroaereus | nitroaereus | nitr...,nitroaereus,True,[no answer],False,"In the late 17th century, Robert Boyle proved ..."
1,2,5726545f708984140094c2a5,"The legislative body, the Council, are made up...",different ministers of the member states | min...,[no answer],False,[no answer],False,The second main legislative body is the Counci...
2,3,5ad3ff1b604f3c001a3ffc74,What revolution was fought in the 1899's?,[no answer],[no answer],True,[no answer],True,The French Wars of Religion in the 16th centur...
3,4,5727e21e4b864d1900163f36,What Shakespeare Scholar is a faculty member a...,Stephen Greenblatt | Stephen Greenblatt | Step...,[no answer],False,[no answer],False,Harvard's faculty includes scholars such as bi...
4,5,5711163bb654c5140001fb13,What present day county is New Rochelle in?,Westchester | Westchester | Westchester,[no answer],False,[no answer],False,"New Rochelle, located in the county of Westche..."
5,6,57339dd94776f41900660ecd,How man people gather along the banks of the V...,thousands | thousands | thousands,thousands of people on the banks of the Vistul...,False,thousands,True,Several commemorative events take place every ...
6,7,572fdd03a23a5019007fca9e,What are MPs unable to vote upon?,domestic legislation of the Scottish Parliamen...,[no answer],False,[no answer],False,A procedural consequence of the establishment ...
7,8,5a892d303b2508001a72a4ee,How many modern types of algorithm tests for g...,[no answer],[no answer],True,[no answer],True,Modern primality tests for general numbers n c...
8,9,5ad04f7977cf76001a686fbb,What NLH teams are from Southern California?,[no answer],[no answer],True,[no answer],True,Professional sports teams in Southern Californ...
9,10,5ad15170645df0001a2d1735,What was not central to European development s...,[no answer],free movement and trade,False,[no answer],True,"While the concept of a ""social market economy""..."


## After This Passes

Use the same functions for nested 10%, 30%, and 50% subsets. The reader models do not need to change when the passage provider switches from SQuAD context to retrieved text.